# Prediction Timeline Viewer

Inspect 2023-2024 test predictions against measured SCADA values, status Warning/Stop intervals, manual event intervals, and alarm points from residual baseline, SCC, or CUSUM.


In [1]:
from pathlib import Path
import sys

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.read_event_dataset import read_event_dataset
from src.data.read_status import read_status_folder

EXPERIMENTS_DIR = PROJECT_ROOT / "results" / "kelmarsh" / "experiments"
STATUS_DIR = PROJECT_ROOT / "data" / "raw" / "kelmarsh" / "status"
EVENT_PATH = PROJECT_ROOT / "data" / "raw" / "kelmarsh" / "auxiliary" / "kelmarsh_event_dataset.csv"
FIGURES_DIR = PROJECT_ROOT / "results" / "kelmarsh" / "figures"
TURBINES = [f"Kelmarsh_{i}" for i in range(1, 7)]
YEARS = [2023, 2024]
MONTHS = list(range(1, 13))


In [2]:
def list_experiments():
    if not EXPERIMENTS_DIR.exists():
        return []
    return sorted(p.name for p in EXPERIMENTS_DIR.iterdir() if (p / "metadata.json").exists())


DETECTION_METHODS = {
    "residual_baseline": {"label": "Residual baseline", "prefix": "residual_baseline"},
    "scc": {"label": "SCC", "prefix": "scc"},
    "cusum": {"label": "CUSUM", "prefix": "cusum"},
    "bcad": {"label": "BCAD", "prefix": "bcad"},
}


def method_prefix(method):
    return DETECTION_METHODS[method]["prefix"]


def list_detection_settings(run_id, method):
    run_dir = EXPERIMENTS_DIR / run_id
    prefix = method_prefix(method)
    files = sorted(run_dir.glob(f"{prefix}_thresholds_*.csv"))
    settings = [p.stem.replace(f"{prefix}_thresholds_", "") for p in files]
    return [setting for setting in settings if "_v" not in setting]


def threshold_path(run_id, method, setting):
    prefix = method_prefix(method)
    return EXPERIMENTS_DIR / run_id / f"{prefix}_thresholds_{setting}.csv"


def detection_path(run_id, method, setting):
    prefix = method_prefix(method)
    return EXPERIMENTS_DIR / run_id / f"{prefix}_detections_{setting}.csv"


def prediction_path(run_id):
    return EXPERIMENTS_DIR / run_id / "test_2023_2024_predictions.csv"


def target_options(run_id, method, setting):
    path = threshold_path(run_id, method, setting)
    if not path.exists():
        return []
    return pd.read_csv(path)["target"].dropna().astype(str).tolist()


def resolve_prediction_column(columns, prefix, target):
    exact = f"{prefix}::{target}"
    if exact in columns:
        return exact
    target_base = target.split("(", 1)[0].strip()
    for col in columns:
        if col.startswith(f"{prefix}::") and col.split("::", 1)[1].split("(", 1)[0].strip() == target_base:
            return col
    raise KeyError(f"Could not find {prefix} column for target: {target}")


def selected_threshold(run_id, method, setting, target):
    thresholds = pd.read_csv(threshold_path(run_id, method, setting))
    row = thresholds.loc[thresholds["target"].astype(str) == str(target)]
    if row.empty:
        target_base = str(target).split("(", 1)[0].strip()
        row = thresholds.loc[thresholds["target"].astype(str).str.split("(", n=1).str[0].str.strip() == target_base]
    if row.empty:
        raise ValueError(f"No threshold found for target: {target}")
    if "threshold" in row.columns:
        return float(row.iloc[0]["threshold"])
    return None


runs = list_experiments()
if not runs:
    raise FileNotFoundError(f"No experiment folders found in {EXPERIMENTS_DIR}")
DEFAULT_RUN = runs[-1]
DEFAULT_METHOD = "residual_baseline"
DEFAULT_SETTINGS = list_detection_settings(DEFAULT_RUN, DEFAULT_METHOD)
if not DEFAULT_SETTINGS:
    raise FileNotFoundError(f"No detection threshold files found for {DEFAULT_RUN}")
DEFAULT_SETTING = DEFAULT_SETTINGS[-1]
DEFAULT_TARGETS = target_options(DEFAULT_RUN, DEFAULT_METHOD, DEFAULT_SETTING)
DEFAULT_TARGET = DEFAULT_TARGETS[0]

print("Default run:", DEFAULT_RUN)
print("Default method:", DETECTION_METHODS[DEFAULT_METHOD]["label"])
print("Default setting:", DEFAULT_SETTING)
print("Targets:", DEFAULT_TARGETS)


Default run: all6_multi3_seq12_h64_l1_bs64_lr1e-03_wd0_do0_e60_p8_seed42
Default method: Residual baseline
Default setting: q995_c6
Targets: ['Generator bearing front temperature (°C)', 'Stator temperature 1 (°C)', 'Rear bearing temperature (°C)']


In [3]:
def month_bounds(year, month):
    start = pd.Timestamp(year=int(year), month=int(month), day=1)
    end = start + pd.offsets.MonthBegin(1)
    return start, end


def load_month_predictions(run_id, turbine_id, target, year, month, chunksize=200_000):
    path = prediction_path(run_id)
    if not path.exists():
        raise FileNotFoundError(f"Missing prediction file: {path}")

    header = pd.read_csv(path, nrows=0)
    columns = header.columns.tolist()
    true_col = resolve_prediction_column(columns, "y_true", target)
    pred_col = resolve_prediction_column(columns, "y_pred", target)
    residual_col = resolve_prediction_column(columns, "residual", target)
    usecols = ["turbine_id", "Date and time", true_col, pred_col, residual_col]
    start, end = month_bounds(year, month)

    parts = []
    for chunk in pd.read_csv(path, usecols=usecols, chunksize=chunksize):
        chunk["Date and time"] = pd.to_datetime(chunk["Date and time"], errors="coerce")
        mask = (
            (chunk["turbine_id"] == turbine_id)
            & chunk["Date and time"].notna()
            & (chunk["Date and time"] >= start)
            & (chunk["Date and time"] < end)
        )
        if mask.any():
            parts.append(chunk.loc[mask].copy())

    if not parts:
        return pd.DataFrame(columns=["Date and time", "measured", "predicted", "residual"])

    out = pd.concat(parts, ignore_index=True).sort_values("Date and time")
    out = out.rename(columns={true_col: "measured", pred_col: "predicted", residual_col: "residual"})
    return out[["Date and time", "measured", "predicted", "residual"]]



def load_month_alarm_points(run_id, method, setting, turbine_id, target, year, month, chunksize=200_000):
    path = detection_path(run_id, method, setting)
    if not path.exists():
        return pd.DataFrame(columns=["Date and time", "is_anomaly", "is_alarm"])
    start, end = month_bounds(year, month)
    usecols = ["turbine_id", "Date and time", "target", "is_anomaly", "is_alarm"]
    parts = []
    for chunk in pd.read_csv(path, usecols=usecols, chunksize=chunksize):
        chunk["Date and time"] = pd.to_datetime(chunk["Date and time"], errors="coerce")
        mask = (
            (chunk["turbine_id"] == turbine_id)
            & chunk["target"].astype(str).eq(str(target))
            & chunk["Date and time"].notna()
            & (chunk["Date and time"] >= start)
            & (chunk["Date and time"] < end)
        )
        if mask.any():
            parts.append(chunk.loc[mask, ["Date and time", "is_anomaly", "is_alarm"]].copy())
    if not parts:
        return pd.DataFrame(columns=["Date and time", "is_anomaly", "is_alarm"])
    return pd.concat(parts, ignore_index=True).sort_values("Date and time")


def clip_interval(start, end, plot_start, plot_end):
    return max(start, plot_start), min(end, plot_end)


def load_status_intervals(turbine_id, year, month):
    folder = STATUS_DIR / turbine_id
    if not folder.exists():
        return {"Warning": [], "Stop": []}
    start, end = month_bounds(year, month)
    status = read_status_folder(folder, turbine_id=turbine_id)
    status["Status"] = status["Status"].astype(str).str.strip()
    intervals = {"Warning": [], "Stop": []}
    for status_name in intervals:
        part = status[
            status["Status"].str.casefold().eq(status_name.casefold())
            & (status["Timestamp start"] < end)
            & (status["Timestamp end"] >= start)
        ].copy()
        for _, row in part.iterrows():
            s, e = clip_interval(row["Timestamp start"], row["Timestamp end"], start, end)
            intervals[status_name].append((s, e, row.get("Message", ""), row.get("IEC category", "")))
    return intervals


def load_manual_event_intervals(turbine_id, year, month):
    if not EVENT_PATH.exists():
        return []
    start, end = month_bounds(year, month)
    turbine_num = turbine_id.split("_")[-1]
    events = read_event_dataset(EVENT_PATH)
    events = events[
        events["Turbine"].astype(str).str.strip().eq(turbine_num)
        & (events["Timestamp start"] < end)
        & (events["Timestamp end"] >= start)
    ].copy()
    intervals = []
    for _, row in events.iterrows():
        s, e = clip_interval(row["Timestamp start"], row["Timestamp end"], start, end)
        label = f"{row.get('Category', '')} / {row.get('Subcategory', '')}".strip(" / ")
        intervals.append((s, e, label))
    return intervals


In [4]:
def plot_prediction_timeline(run_id, method, setting, turbine_id, target, year, month, show_outliers):
    clear_output(wait=True)
    start, end = month_bounds(year, month)
    pred = load_month_predictions(run_id, turbine_id, target, year, month)
    if pred.empty:
        print(f"No prediction data for {turbine_id}, {year}-{int(month):02d}, {target}")
        return

    threshold = selected_threshold(run_id, method, setting, target)
    alarm_points = load_month_alarm_points(run_id, method, setting, turbine_id, target, year, month)
    status_intervals = load_status_intervals(turbine_id, year, month)
    manual_intervals = load_manual_event_intervals(turbine_id, year, month)

    fig, ax = plt.subplots(figsize=(16, 6), facecolor="white")
    ax.set_facecolor("white")

    # Draw status first. Manual events are drawn last so their color has priority in overlaps.
    for status_name, color, alpha in [
        ("Warning", "#2f80ed", 0.14),
        ("Stop", "#ffd500", 0.20),
    ]:
        for i, (s, e, message, iec) in enumerate(status_intervals[status_name]):
            ax.axvspan(s, e, color=color, alpha=alpha, label=f"Status {status_name}" if i == 0 else None)

    ax.plot(pred["Date and time"], pred["measured"], color="#0057b8", linewidth=1.15, label="Measured")
    ax.plot(pred["Date and time"], pred["predicted"], color="#f28e2b", linewidth=1.15, label="Predicted")

    alarm_mask = pd.Series(False, index=pred.index)
    if not alarm_points.empty:
        alarm_times = set(alarm_points.loc[alarm_points["is_alarm"].fillna(False).astype(bool), "Date and time"])
        alarm_mask = pred["Date and time"].isin(alarm_times)
    if show_outliers and alarm_mask.any():
        ax.scatter(
            pred.loc[alarm_mask, "Date and time"],
            pred.loc[alarm_mask, "measured"],
            s=16,
            color="#d00000",
            label="Detection alarm point",
            zorder=5,
        )

    for i, (s, e, label) in enumerate(manual_intervals):
        ax.axvspan(s, e, color="#d00000", alpha=0.26, label="Manual event" if i == 0 else None)

    ax.set_xlim(start, end)
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
    ax.grid(axis="x", color="#d0d0d0", alpha=0.6)
    ax.grid(axis="y", color="#e2e2e2", alpha=0.45)
    ax.set_title(f"{turbine_id} | {target} | measured vs predicted | {year}-{int(month):02d} | {DETECTION_METHODS[method]['label']} | {setting}")
    ax.set_xlabel("Time")
    ax.set_ylabel(target)
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=4)
    plt.tight_layout()
    plt.show()

    summary = pd.DataFrame([{
        "method": DETECTION_METHODS[method]["label"],
        "turbine": turbine_id,
        "year_month": f"{year}-{int(month):02d}",
        "target": target,
        "threshold_or_decision_h": threshold,
        "prediction_rows": len(pred),
        "alarm_points": int(alarm_mask.sum()),
        "alarm_point_ratio": float(alarm_mask.mean()),
        "warning_intervals": len(status_intervals["Warning"]),
        "stop_intervals": len(status_intervals["Stop"]),
        "manual_event_intervals": len(manual_intervals),
    }])
    display(summary)


In [5]:
method_dropdown = widgets.Dropdown(
    options=[(v["label"], k) for k, v in DETECTION_METHODS.items()],
    value=DEFAULT_METHOD,
    description="Method",
)
run_dropdown = widgets.Dropdown(options=runs, value=DEFAULT_RUN, description="Run")
setting_dropdown = widgets.Dropdown(options=DEFAULT_SETTINGS, value=DEFAULT_SETTING, description="Setting")
turbine_dropdown = widgets.Dropdown(options=TURBINES, value="Kelmarsh_1", description="Turbine")
target_dropdown = widgets.Dropdown(options=DEFAULT_TARGETS, value=DEFAULT_TARGET, description="Target")
year_dropdown = widgets.Dropdown(options=YEARS, value=2023, description="Year")
month_dropdown = widgets.Dropdown(options=MONTHS, value=1, description="Month")
show_outliers_checkbox = widgets.Checkbox(value=True, description="Show alarm points")


def refresh_settings(*_):
    settings = list_detection_settings(run_dropdown.value, method_dropdown.value)
    setting_dropdown.options = settings
    if settings:
        setting_dropdown.value = settings[-1]


def refresh_targets(*_):
    targets = target_options(run_dropdown.value, method_dropdown.value, setting_dropdown.value)
    target_dropdown.options = targets
    if targets:
        target_dropdown.value = targets[0]


method_dropdown.observe(refresh_settings, names="value")
run_dropdown.observe(refresh_settings, names="value")
setting_dropdown.observe(refresh_targets, names="value")

ui = widgets.VBox([
    widgets.HBox([method_dropdown, run_dropdown, setting_dropdown]),
    widgets.HBox([turbine_dropdown, target_dropdown]),
    widgets.HBox([year_dropdown, month_dropdown, show_outliers_checkbox]),
])

out = widgets.interactive_output(
    plot_prediction_timeline,
    {
        "run_id": run_dropdown,
        "method": method_dropdown,
        "setting": setting_dropdown,
        "turbine_id": turbine_dropdown,
        "target": target_dropdown,
        "year": year_dropdown,
        "month": month_dropdown,
        "show_outliers": show_outliers_checkbox,
    },
)

display(ui, out)


Output()